In [ ]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================

!pip install -q transformers accelerate safetensors opencv-python pillow

# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =========================================================
# IMPORTS
# =========================================================

import os
import cv2
import json
import shutil
import logging
import numpy as np
import torch

from PIL import Image
from datetime import datetime

from transformers import (
    AutoModelForZeroShotObjectDetection,
    AutoProcessor
)

# =========================================================
# BASE PATH
# =========================================================

BASE_PATH = "/content/drive/MyDrive/indian_helmet_detection_dataset"

VIDEO_PATH           = os.path.join(BASE_PATH, "laketown_15sec.mp4")
OUTPUT_VIDEO_PATH    = "/content/output_inference.avi"
FINAL_SAVE_PATH      = os.path.join(BASE_PATH, "output_inference.avi")

HELMET_CROP_DIR      = os.path.join(BASE_PATH, "helmet_crops")
NO_HELMET_CROP_DIR   = os.path.join(BASE_PATH, "no_helmet_crops")   # NEW
FRAME_DEBUG_DIR      = os.path.join(BASE_PATH, "groundingdino_frame_debug_logs")
DEBUG_LOG_PATH       = os.path.join(BASE_PATH, "groundingdino_debug_log.txt")
DETECTIONS_JSON_PATH = os.path.join(BASE_PATH, "groundingdino_detections.json")

for d in [HELMET_CROP_DIR, NO_HELMET_CROP_DIR, FRAME_DEBUG_DIR]:
    os.makedirs(d, exist_ok=True)
    for f in os.listdir(d):
        fp = os.path.join(d, f)
        if os.path.isfile(fp):
            os.remove(fp)

# =========================================================
# LOGGING
# — both file and console use timestamps
# — log_frame() now prints to terminal as well
# =========================================================

logger = logging.getLogger("groundingdino_laketown")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("[%(asctime)s] %(message)s",
                              datefmt="%Y-%m-%d %H:%M:%S")

file_handler = logging.FileHandler(DEBUG_LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()           # terminal output
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"USING DEVICE: {device}")

if device != "cuda":
    raise RuntimeError("GPU not enabled. Runtime > Change runtime type > GPU.")

# =========================================================
# MODEL
# =========================================================

GROUNDINGDINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

logger.info("LOADING PROCESSOR...")
processor = AutoProcessor.from_pretrained(GROUNDINGDINO_MODEL_ID)
logger.info("PROCESSOR LOADED.")

logger.info("LOADING MODEL...")
model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDINGDINO_MODEL_ID)
model.to(device)
model.eval()
logger.info("MODEL LOADED.")

# =========================================================
# SETTINGS
# =========================================================

TEXT_PROMPT     = "head with helmet . human head . "
BOX_THRESHOLD   = 0.5
TEXT_THRESHOLD  = 0.5
INFERENCE_WIDTH = 896

MIN_BOX_SIZE = 35
MAX_BOX_SIZE = 120
MIN_ASPECT   = 0.7
MAX_ASPECT   = 1.3

TRACKER_MAX_FAILS  = 10
TRACKER_IOU_THRESH = 0.3

# =========================================================
# ROI POLYGON
# =========================================================

ROI_POINTS = np.array([
    [0,    1215],
    [297,  1004],
    [2064, 1071],
    [2067, 1510],
    [13,   1515]
], dtype=np.int32)

logger.info(f"ROI POINTS: {ROI_POINTS.tolist()}")

# =========================================================
# OPEN VIDEO
# =========================================================

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise Exception(f"Could not open video: {VIDEO_PATH}")

fps          = int(cap.get(cv2.CAP_PROP_FPS))
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

logger.info(f"VIDEO: {width}x{height} @ {fps}fps, {total_frames} frames")

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out    = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# =========================================================
# HELPERS
# =========================================================

def to_device(inputs):
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}


def post_process_groundingdino(outputs, input_ids, target_sizes):
    try:
        return processor.post_process_grounded_object_detection(
            outputs, input_ids,
            box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
            target_sizes=target_sizes
        )[0]
    except TypeError:
        return processor.post_process_grounded_object_detection(
            outputs, input_ids=input_ids,
            threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
            target_sizes=target_sizes
        )[0]


def iou(a, b):
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    if inter == 0:
        return 0.0
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / max(union, 1)


def make_tracker():
    try:    return cv2.TrackerCSRT_create()
    except: return cv2.TrackerKCF_create()


def save_crop(frame, x1, y1, x2, y2, directory, label, frame_idx, score, ts):
    """
    Save a detection crop.
    Filename encodes: frame index, label, confidence score, wall-clock timestamp.
    Returns the saved path or None if the crop was empty.
    """
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    ts_file = ts.replace(":", "-").replace(" ", "_")
    filename = (
        f"frame_{frame_idx:06d}_{label}_"
        f"conf{score:.2f}_{ts_file}.jpg"
    )
    path = os.path.join(directory, filename)
    cv2.imwrite(path, crop)
    return path

# =========================================================
# COUNTERS / STATE
# =========================================================

frame_count    = 0
crop_count     = 0
all_detections = []

helmet_tracks          = {}
NEXT_TRACK_ID          = 0
MAX_CENTER_DISTANCE    = 50
MIN_CONSECUTIVE_FRAMES = 2

active_trackers = {}
NEXT_CSRT_ID    = 0

# =========================================================
# MAIN LOOP
# =========================================================

while True:

    ret, frame = cap.read()
    if not ret:
        logger.info("VIDEO FINISHED.")
        break

    frame_lines = []
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # log_frame writes to terminal (via logger) AND to the per-frame file
    def log_frame(msg):
        logger.info(msg)                        # → console + main log file (timestamped)
        frame_lines.append(f"[{ts}] {msg}")     # → per-frame debug file

    log_frame(f"================ FRAME {frame_count} ================")

    h, w = frame.shape[:2]
    annotated_frame = frame.copy()
    cv2.polylines(annotated_frame, [ROI_POINTS], True, (255, 0, 0), 3)

    # =================================================
    # STEP 1: UPDATE ACTIVE CSRT TRACKERS
    # =================================================

    to_remove = []
    for tid, td in active_trackers.items():
        ok, bbox = td["tracker"].update(frame)
        if ok:
            tx1 = max(0, int(bbox[0]))
            ty1 = max(0, int(bbox[1]))
            tx2 = min(w, int(bbox[0] + bbox[2]))
            ty2 = min(h, int(bbox[1] + bbox[3]))
            td["box"]   = [tx1, ty1, tx2, ty2]
            td["fails"] = 0
            log_frame(f"TRACKER {tid}: OK -> ({tx1},{ty1},{tx2},{ty2})")

            cv2.rectangle(annotated_frame, (tx1, ty1), (tx2, ty2), (0, 255, 0), 4)
            cv2.putText(annotated_frame, f"helmet [T{tid}]",
                        (tx1, max(30, ty1-10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)

            path = save_crop(frame, tx1, ty1, tx2, ty2,
                             HELMET_CROP_DIR, f"helmet_tracker{tid}",
                             frame_count, td.get("score", 0.0), ts)
            if path:
                log_frame(f"TRACKER CROP SAVED: {os.path.basename(path)}")
                crop_count += 1
        else:
            td["fails"] += 1
            log_frame(f"TRACKER {tid}: FAILED (fails={td['fails']})")
            if td["fails"] >= TRACKER_MAX_FAILS:
                to_remove.append(tid)
                log_frame(f"TRACKER {tid}: RETIRED after {TRACKER_MAX_FAILS} fails")

    for tid in to_remove:
        del active_trackers[tid]

    # =================================================
    # STEP 2: ROI MASK + GROUNDINGDINO INFERENCE
    # =================================================

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [ROI_POINTS], 255)
    roi_frame   = cv2.bitwise_and(frame, frame, mask=mask)

    roi_x, roi_y, roi_w, roi_h = cv2.boundingRect(ROI_POINTS)
    model_frame = roi_frame[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]
    model_h, model_w = model_frame.shape[:2]

    if INFERENCE_WIDTH and model_w > INFERENCE_WIDTH:
        scale   = INFERENCE_WIDTH / model_w
        infer_w = INFERENCE_WIDTH
        infer_h = max(1, int(round(model_h * scale)))
        inference_frame  = cv2.resize(model_frame, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
        scale_x, scale_y = model_w / infer_w, model_h / infer_h
    else:
        inference_frame  = model_frame
        infer_h, infer_w = model_h, model_w
        scale_x = scale_y = 1.0

    log_frame(f"ROI: x={roi_x} y={roi_y} w={roi_w} h={roi_h} | MODEL INPUT: {infer_w}x{infer_h}")

    pil_frame = Image.fromarray(cv2.cvtColor(inference_frame, cv2.COLOR_BGR2RGB))
    inputs    = processor(images=pil_frame, text=TEXT_PROMPT, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    inputs    = to_device(inputs)

    with torch.no_grad():
        outputs = model(**inputs)

    results   = post_process_groundingdino(
        outputs, input_ids,
        torch.tensor([[infer_h, infer_w]], device=device)
    )

    total_raw        = len(results["scores"])
    total_helmet_det = 0
    total_nohelmet_det = 0                                 # NEW
    log_frame(f"TOTAL RAW DETECTIONS: {total_raw}")

    # =================================================
    # STEP 3: PROCESS DETECTIONS
    # =================================================

    for box_index, (score, label, box) in enumerate(
        zip(results["scores"], results["labels"], results["boxes"])
    ):
        score_value = float(score.item() if hasattr(score, "item") else score)
        label_value = label.item() if hasattr(label, "item") else label
        class_name  = str(label_value)

        bx1, by1, bx2, by2 = box.tolist()
        x1 = int(round((bx1 * scale_x) + roi_x))
        y1 = int(round((by1 * scale_y) + roi_y))
        x2 = int(round((bx2 * scale_x) + roi_x))
        y2 = int(round((by2 * scale_y) + roi_y))

        log_frame(f"RAW #{box_index} | label={label_value} | score={score_value:.4f} | box=({x1},{y1},{x2},{y2})")

        is_helmet    = "helmet" in class_name.lower()
        is_no_helmet = not is_helmet                       # human head = no helmet

        # -----------------------------------------------
        # COMMON FILTERS (ROI + geometry) for both classes
        # -----------------------------------------------

        cx, cy = (x1+x2)//2, (y1+y2)//2
        inside = cv2.pointPolygonTest(ROI_POINTS, (cx, cy), False)
        log_frame(f"  CENTER: ({cx},{cy}) | ROI test: {inside}")
        if inside < 0:
            log_frame("  SKIPPED -> OUTSIDE ROI")
            continue

        box_w, box_h = x2-x1, y2-y1
        if box_w <= 0 or box_h <= 0:
            log_frame("  SKIPPED -> INVALID BOX")
            continue

        aspect_ratio = box_w / box_h
        log_frame(f"  W={box_w} H={box_h} AR={aspect_ratio:.2f}")

        if not (MIN_BOX_SIZE <= box_w <= MAX_BOX_SIZE):
            log_frame("  SKIPPED -> WIDTH OUTLIER")
            continue
        if not (MIN_BOX_SIZE <= box_h <= MAX_BOX_SIZE):
            log_frame("  SKIPPED -> HEIGHT OUTLIER")
            continue
        if not (MIN_ASPECT <= aspect_ratio <= MAX_ASPECT):
            log_frame("  SKIPPED -> ASPECT RATIO OUTLIER")
            continue

        # -----------------------------------------------
        # NO_HELMET BRANCH  (human head)
        # Save crop + draw red box; no tracker needed
        # -----------------------------------------------

        if is_no_helmet:
            x1c, y1c = max(0, x1), max(0, y1)
            x2c, y2c = min(w, x2), min(h, y2)

            path = save_crop(frame, x1c, y1c, x2c, y2c,
                             NO_HELMET_CROP_DIR, "no_helmet",
                             frame_count, score_value, ts)
            if path:
                log_frame(f"  NO_HELMET CROP SAVED: {os.path.basename(path)} | conf={score_value:.4f} | ts={ts}")
                crop_count += 1
            else:
                log_frame("  NO_HELMET CROP EMPTY.")

            cv2.rectangle(annotated_frame, (x1c, y1c), (x2c, y2c), (0, 0, 255), 6)
            cv2.putText(annotated_frame, f"no_helmet {score_value:.2f}",
                        (x1c, max(30, y1c-10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 4)

            all_detections.append({
                "frame": frame_count, "timestamp": ts,
                "score": score_value, "label": "no_helmet",
                "box": [x1c, y1c, x2c, y2c], "center": [cx, cy],
                "crop_path": path
            })

            total_nohelmet_det += 1
            log_frame(f"  NO_HELMET DETECTION ACCEPTED | conf={score_value:.4f}")
            continue

        # -----------------------------------------------
        # HELMET BRANCH
        # -----------------------------------------------

        # Temporal stability
        matched_track_id = None
        for track_id, track_data in helmet_tracks.items():
            prev_cx, prev_cy = track_data["center"]
            if np.sqrt((cx-prev_cx)**2 + (cy-prev_cy)**2) < MAX_CENTER_DISTANCE:
                matched_track_id = track_id
                break

        if matched_track_id is not None:
            helmet_tracks[matched_track_id]["center"] = (cx, cy)
            helmet_tracks[matched_track_id]["count"]  += 1
        else:
            matched_track_id = NEXT_TRACK_ID
            helmet_tracks[matched_track_id] = {"center": (cx, cy), "count": 1}
            NEXT_TRACK_ID += 1

        count_now = helmet_tracks[matched_track_id]["count"]
        log_frame(f"  TEMPORAL: track_id={matched_track_id} count={count_now}/{MIN_CONSECUTIVE_FRAMES}")

        if count_now < MIN_CONSECUTIVE_FRAMES:
            log_frame("  SKIPPED -> NOT STABLE YET")
            continue

        # Match / spawn CSRT tracker
        det_box = [x1, y1, x2, y2]
        best_tid, best_iou_val = None, 0.0
        for tid, td in active_trackers.items():
            v = iou(det_box, td["box"])
            if v > best_iou_val:
                best_iou_val, best_tid = v, tid

        if best_iou_val >= TRACKER_IOU_THRESH and best_tid is not None:
            t = make_tracker()
            t.init(frame, (x1, y1, x2-x1, y2-y1))
            active_trackers[best_tid].update({"tracker": t, "box": det_box, "fails": 0, "score": score_value})
            log_frame(f"  TRACKER {best_tid}: RE-INIT (IoU={best_iou_val:.2f})")
        else:
            t = make_tracker()
            t.init(frame, (x1, y1, x2-x1, y2-y1))
            active_trackers[NEXT_CSRT_ID] = {"tracker": t, "box": det_box, "fails": 0, "score": score_value}
            log_frame(f"  TRACKER {NEXT_CSRT_ID}: SPAWNED")
            NEXT_CSRT_ID += 1

        # Draw + save crop
        x1c, y1c = max(0, x1), max(0, y1)
        x2c, y2c = min(w, x2), min(h, y2)

        cv2.rectangle(annotated_frame, (x1c, y1c), (x2c, y2c), (0, 255, 0), 6)
        cv2.putText(annotated_frame, f"helmet {score_value:.2f}",
                    (x1c, max(30, y1c-10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 4)

        path = save_crop(frame, x1c, y1c, x2c, y2c,
                         HELMET_CROP_DIR, "helmet",
                         frame_count, score_value, ts)
        if path:
            log_frame(f"  HELMET CROP SAVED: {os.path.basename(path)} | conf={score_value:.4f} | ts={ts}")
            crop_count += 1
        else:
            log_frame("  HELMET CROP EMPTY.")

        all_detections.append({
            "frame": frame_count, "timestamp": ts,
            "score": score_value, "label": "helmet",
            "box": [x1c, y1c, x2c, y2c], "center": [cx, cy],
            "crop_path": path
        })

        total_helmet_det += 1
        log_frame(f"  HELMET DETECTION ACCEPTED | conf={score_value:.4f}")

    # =====================================================
    # FRAME SUMMARY
    # =====================================================

    log_frame(
        f"FRAME {frame_count} SUMMARY | "
        f"RAW: {total_raw} | "
        f"HELMET: {total_helmet_det} | "
        f"NO_HELMET: {total_nohelmet_det} | "
        f"TRACKERS: {len(active_trackers)} | "
        f"TOTAL CROPS: {crop_count}"
    )

    with open(os.path.join(FRAME_DEBUG_DIR, f"frame_{frame_count:06d}_debug.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(frame_lines) + "\n")

    out.write(annotated_frame)
    frame_count += 1

    if frame_count % 10 == 0:
        logger.info(f"PROCESSED {frame_count}/{total_frames} FRAMES")

# =========================================================
# RELEASE + SAVE
# =========================================================

cap.release()
out.release()

shutil.copy(OUTPUT_VIDEO_PATH, FINAL_SAVE_PATH)

with open(DETECTIONS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_detections, f, indent=2)

logger.info("INFERENCE COMPLETED SUCCESSFULLY")
logger.info(f"OUTPUT VIDEO:    {FINAL_SAVE_PATH}")
logger.info(f"TOTAL CROPS:     {crop_count}")
logger.info(f"HELMET CROPS:    {HELMET_CROP_DIR}")
logger.info(f"NO_HELMET CROPS: {NO_HELMET_CROP_DIR}")
logger.info(f"DETECTIONS JSON: {DETECTIONS_JSON_PATH}")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# # =========================================================
# # INSTALL DEPENDENCIES
# # =========================================================

# !pip install -q transformers accelerate safetensors opencv-contrib-python pillow

# # =========================================================
# # IMPORTS
# # =========================================================

# import os
# import cv2
# import json
# import shutil
# import logging
# import numpy as np
# import torch

# from PIL import Image
# from datetime import datetime

# from transformers import (
#     AutoModelForZeroShotObjectDetection,
#     AutoProcessor
# )

# # =========================================================
# # BASE PATH
# # =========================================================

# BASE_PATH         = "/content/drive/MyDrive/indian_helmet_detection_dataset"
# OUTPUT_VIDEO_PATH = "/content/output_inference.avi"

# VIDEO_PATH           = os.path.join(BASE_PATH, "laketown_15sec.mp4")
# FINAL_SAVE_PATH      = os.path.join(BASE_PATH, "output_inference.avi")
# HELMET_CROP_DIR      = os.path.join(BASE_PATH, "helmet_crops")
# NO_HELMET_CROP_DIR   = os.path.join(BASE_PATH, "no_helmet_crops")
# FRAME_DEBUG_DIR      = os.path.join(BASE_PATH, "groundingdino_frame_debug_logs")
# DEBUG_LOG_PATH       = os.path.join(BASE_PATH, "groundingdino_debug_log.txt")
# DETECTIONS_JSON_PATH = os.path.join(BASE_PATH, "groundingdino_detections.json")

# for d in [HELMET_CROP_DIR, NO_HELMET_CROP_DIR, FRAME_DEBUG_DIR]:
#     os.makedirs(d, exist_ok=True)
#     for f in os.listdir(d):
#         fp = os.path.join(d, f)
#         if os.path.isfile(fp):
#             os.remove(fp)

# # =========================================================
# # LOGGING
# # =========================================================

# logger = logging.getLogger("groundingdino_laketown")
# logger.setLevel(logging.INFO)
# logger.handlers.clear()

# formatter = logging.Formatter("[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

# file_handler = logging.FileHandler(DEBUG_LOG_PATH, mode="w", encoding="utf-8")
# file_handler.setFormatter(formatter)
# logger.addHandler(file_handler)

# console_handler = logging.StreamHandler()
# console_handler.setFormatter(formatter)
# logger.addHandler(console_handler)

# # =========================================================
# # DEVICE
# # =========================================================

# device = "cuda" if torch.cuda.is_available() else "cpu"
# logger.info(f"USING DEVICE: {device}")

# if device != "cuda":
#     raise RuntimeError("GPU not enabled. Runtime > Change runtime type > GPU.")

# # =========================================================
# # MODEL
# # =========================================================

# GROUNDINGDINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

# logger.info("LOADING PROCESSOR...")
# processor = AutoProcessor.from_pretrained(GROUNDINGDINO_MODEL_ID)
# logger.info("PROCESSOR LOADED.")

# logger.info("LOADING MODEL...")
# model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDINGDINO_MODEL_ID)
# model.to(device)
# model.eval()
# logger.info("MODEL LOADED.")

# # =========================================================
# # SETTINGS
# # =========================================================

# TEXT_PROMPT     = "head with helmet . human head . "
# BOX_THRESHOLD   = 0.5
# TEXT_THRESHOLD  = 0.5
# INFERENCE_WIDTH = 896

# MIN_BOX_SIZE = 35
# MAX_BOX_SIZE = 120
# MIN_ASPECT   = 0.7
# MAX_ASPECT   = 1.3

# TRACKER_MAX_FAILS  = 10
# TRACKER_IOU_THRESH = 0.3

# # =========================================================
# # ROI POLYGON
# # =========================================================

# ROI_POINTS = np.array([
#     [0,    1215],
#     [297,  1004],
#     [2064, 1071],
#     [2067, 1510],
#     [13,   1515]
# ], dtype=np.int32)

# logger.info(f"ROI POINTS: {ROI_POINTS.tolist()}")

# # =========================================================
# # OPEN VIDEO
# # =========================================================

# cap = cv2.VideoCapture(VIDEO_PATH)
# if not cap.isOpened():
#     raise Exception(f"Could not open video: {VIDEO_PATH}")

# fps          = int(cap.get(cv2.CAP_PROP_FPS))
# width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# logger.info(f"VIDEO: {width}x{height} @ {fps}fps, {total_frames} frames")

# fourcc = cv2.VideoWriter_fourcc(*'XVID')
# out    = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

# # =========================================================
# # HELPERS
# # =========================================================

# def to_device(inputs):
#     return {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}


# def post_process_groundingdino(outputs, input_ids, target_sizes):
#     try:
#         return processor.post_process_grounded_object_detection(
#             outputs, input_ids,
#             box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
#             target_sizes=target_sizes
#         )[0]
#     except TypeError:
#         return processor.post_process_grounded_object_detection(
#             outputs, input_ids=input_ids,
#             threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
#             target_sizes=target_sizes
#         )[0]


# def iou(a, b):
#     xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
#     xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
#     inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
#     if inter == 0:
#         return 0.0
#     union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
#     return inter / max(union, 1)


# def make_tracker():
#     # opencv-contrib-python moved trackers to cv2.legacy in newer builds
#     for fn in [
#         lambda: cv2.legacy.TrackerCSRT_create(),
#         lambda: cv2.legacy.TrackerKCF_create(),
#         lambda: cv2.TrackerCSRT_create(),
#         lambda: cv2.TrackerKCF_create(),
#     ]:
#         try:
#             return fn()
#         except AttributeError:
#             continue
#     raise RuntimeError("No CSRT/KCF tracker found. Ensure opencv-contrib-python is installed.")


# def save_crop(frame, x1, y1, x2, y2, directory, label, frame_idx, score, ts):
#     crop = frame[y1:y2, x1:x2]
#     if crop.size == 0:
#         return None
#     ts_file  = ts.replace(":", "-").replace(" ", "_")
#     filename = f"frame_{frame_idx:06d}_{label}_conf{score:.2f}_{ts_file}.jpg"
#     path     = os.path.join(directory, filename)
#     cv2.imwrite(path, crop)
#     return path

# # =========================================================
# # COUNTERS / STATE
# # =========================================================

# frame_count    = 0
# crop_count     = 0
# all_detections = []

# helmet_tracks          = {}
# NEXT_TRACK_ID          = 0
# MAX_CENTER_DISTANCE    = 50
# MIN_CONSECUTIVE_FRAMES = 2

# active_trackers = {}
# NEXT_CSRT_ID    = 0

# # =========================================================
# # MAIN LOOP
# # =========================================================

# while True:

#     ret, frame = cap.read()
#     if not ret:
#         logger.info("VIDEO FINISHED.")
#         break

#     frame_lines = []
#     ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#     def log_frame(msg):
#         logger.info(msg)                         # → console + main log (timestamped by formatter)
#         frame_lines.append(f"[{ts}] {msg}")      # → per-frame debug file

#     log_frame(f"================ FRAME {frame_count} ================")

#     h, w = frame.shape[:2]
#     annotated_frame = frame.copy()
#     cv2.polylines(annotated_frame, [ROI_POINTS], True, (255, 0, 0), 3)

#     # =================================================
#     # STEP 1: UPDATE ACTIVE CSRT TRACKERS
#     # =================================================

#     to_remove = []
#     for tid, td in active_trackers.items():
#         ok, bbox = td["tracker"].update(frame)
#         if ok:
#             tx1 = max(0, int(bbox[0]))
#             ty1 = max(0, int(bbox[1]))
#             tx2 = min(w, int(bbox[0] + bbox[2]))
#             ty2 = min(h, int(bbox[1] + bbox[3]))
#             td["box"]   = [tx1, ty1, tx2, ty2]
#             td["fails"] = 0
#             log_frame(f"TRACKER {tid}: OK -> ({tx1},{ty1},{tx2},{ty2})")

#             cv2.rectangle(annotated_frame, (tx1, ty1), (tx2, ty2), (0, 255, 0), 4)
#             cv2.putText(annotated_frame, f"helmet [T{tid}]",
#                         (tx1, max(30, ty1-10)),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)

#             path = save_crop(frame, tx1, ty1, tx2, ty2,
#                              HELMET_CROP_DIR, f"helmet_tracker{tid}",
#                              frame_count, td.get("score", 0.0), ts)
#             if path:
#                 log_frame(f"TRACKER CROP SAVED: {os.path.basename(path)}")
#                 crop_count += 1
#         else:
#             td["fails"] += 1
#             log_frame(f"TRACKER {tid}: FAILED (fails={td['fails']})")
#             if td["fails"] >= TRACKER_MAX_FAILS:
#                 to_remove.append(tid)
#                 log_frame(f"TRACKER {tid}: RETIRED after {TRACKER_MAX_FAILS} fails")

#     for tid in to_remove:
#         del active_trackers[tid]

#     # =================================================
#     # STEP 2: ROI MASK + GROUNDINGDINO INFERENCE
#     # =================================================

#     mask = np.zeros((h, w), dtype=np.uint8)
#     cv2.fillPoly(mask, [ROI_POINTS], 255)
#     roi_frame   = cv2.bitwise_and(frame, frame, mask=mask)

#     roi_x, roi_y, roi_w, roi_h = cv2.boundingRect(ROI_POINTS)
#     model_frame = roi_frame[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]
#     model_h, model_w = model_frame.shape[:2]

#     if INFERENCE_WIDTH and model_w > INFERENCE_WIDTH:
#         scale   = INFERENCE_WIDTH / model_w
#         infer_w = INFERENCE_WIDTH
#         infer_h = max(1, int(round(model_h * scale)))
#         inference_frame  = cv2.resize(model_frame, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
#         scale_x, scale_y = model_w / infer_w, model_h / infer_h
#     else:
#         inference_frame  = model_frame
#         infer_h, infer_w = model_h, model_w
#         scale_x = scale_y = 1.0

#     log_frame(f"ROI: x={roi_x} y={roi_y} w={roi_w} h={roi_h} | MODEL INPUT: {infer_w}x{infer_h}")

#     pil_frame = Image.fromarray(cv2.cvtColor(inference_frame, cv2.COLOR_BGR2RGB))
#     inputs    = processor(images=pil_frame, text=TEXT_PROMPT, return_tensors="pt")
#     input_ids = inputs.input_ids.to(device)
#     inputs    = to_device(inputs)

#     with torch.no_grad():
#         outputs = model(**inputs)

#     results = post_process_groundingdino(
#         outputs, input_ids,
#         torch.tensor([[infer_h, infer_w]], device=device)
#     )

#     total_raw          = len(results["scores"])
#     total_helmet_det   = 0
#     total_nohelmet_det = 0
#     log_frame(f"TOTAL RAW DETECTIONS: {total_raw}")

#     # Use text_labels if available (v4.51+), else fall back to labels
#     label_key = "text_labels" if "text_labels" in results else "labels"

#     # =================================================
#     # STEP 3: PROCESS DETECTIONS
#     # =================================================

#     for box_index, (score, label, box) in enumerate(
#         zip(results["scores"], results[label_key], results["boxes"])
#     ):
#         score_value = float(score.item() if hasattr(score, "item") else score)
#         class_name  = str(label) if isinstance(label, str) else str(
#             label.item() if hasattr(label, "item") else label
#         )

#         bx1, by1, bx2, by2 = box.tolist()
#         x1 = int(round((bx1 * scale_x) + roi_x))
#         y1 = int(round((by1 * scale_y) + roi_y))
#         x2 = int(round((bx2 * scale_x) + roi_x))
#         y2 = int(round((by2 * scale_y) + roi_y))

#         log_frame(f"RAW #{box_index} | label={class_name} | score={score_value:.4f} | box=({x1},{y1},{x2},{y2})")

#         is_helmet = "helmet" in class_name.lower()

#         # -----------------------------------------------
#         # COMMON FILTERS: ROI + geometry
#         # -----------------------------------------------

#         cx, cy = (x1+x2)//2, (y1+y2)//2
#         inside = cv2.pointPolygonTest(ROI_POINTS, (cx, cy), False)
#         log_frame(f"  CENTER: ({cx},{cy}) | ROI test: {inside}")
#         if inside < 0:
#             log_frame("  SKIPPED -> OUTSIDE ROI")
#             continue

#         box_w, box_h = x2-x1, y2-y1
#         if box_w <= 0 or box_h <= 0:
#             log_frame("  SKIPPED -> INVALID BOX")
#             continue

#         aspect_ratio = box_w / box_h
#         log_frame(f"  W={box_w} H={box_h} AR={aspect_ratio:.2f}")

#         if not (MIN_BOX_SIZE <= box_w <= MAX_BOX_SIZE):
#             log_frame("  SKIPPED -> WIDTH OUTLIER")
#             continue
#         if not (MIN_BOX_SIZE <= box_h <= MAX_BOX_SIZE):
#             log_frame("  SKIPPED -> HEIGHT OUTLIER")
#             continue
#         if not (MIN_ASPECT <= aspect_ratio <= MAX_ASPECT):
#             log_frame("  SKIPPED -> ASPECT RATIO OUTLIER")
#             continue

#         x1c, y1c = max(0, x1), max(0, y1)
#         x2c, y2c = min(w, x2), min(h, y2)

#         # -----------------------------------------------
#         # NO_HELMET BRANCH
#         # -----------------------------------------------

#         if not is_helmet:
#             path = save_crop(frame, x1c, y1c, x2c, y2c,
#                              NO_HELMET_CROP_DIR, "no_helmet",
#                              frame_count, score_value, ts)
#             if path:
#                 log_frame(f"  NO_HELMET CROP SAVED: {os.path.basename(path)} | conf={score_value:.4f}")
#                 crop_count += 1
#             else:
#                 log_frame("  NO_HELMET CROP EMPTY.")

#             cv2.rectangle(annotated_frame, (x1c, y1c), (x2c, y2c), (0, 0, 255), 6)
#             cv2.putText(annotated_frame, f"no_helmet {score_value:.2f}",
#                         (x1c, max(30, y1c-10)),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 4)

#             all_detections.append({
#                 "frame": frame_count, "timestamp": ts,
#                 "score": score_value, "label": "no_helmet",
#                 "box": [x1c, y1c, x2c, y2c], "center": [cx, cy], "crop_path": path
#             })
#             total_nohelmet_det += 1
#             log_frame(f"  NO_HELMET DETECTION ACCEPTED | conf={score_value:.4f}")
#             continue

#         # -----------------------------------------------
#         # HELMET BRANCH: temporal stability
#         # -----------------------------------------------

#         matched_track_id = None
#         for track_id, track_data in helmet_tracks.items():
#             prev_cx, prev_cy = track_data["center"]
#             if np.sqrt((cx-prev_cx)**2 + (cy-prev_cy)**2) < MAX_CENTER_DISTANCE:
#                 matched_track_id = track_id
#                 break

#         if matched_track_id is not None:
#             helmet_tracks[matched_track_id]["center"] = (cx, cy)
#             helmet_tracks[matched_track_id]["count"]  += 1
#         else:
#             matched_track_id = NEXT_TRACK_ID
#             helmet_tracks[matched_track_id] = {"center": (cx, cy), "count": 1}
#             NEXT_TRACK_ID += 1

#         count_now = helmet_tracks[matched_track_id]["count"]
#         log_frame(f"  TEMPORAL: track_id={matched_track_id} count={count_now}/{MIN_CONSECUTIVE_FRAMES}")

#         if count_now < MIN_CONSECUTIVE_FRAMES:
#             log_frame("  SKIPPED -> NOT STABLE YET")
#             continue

#         # Match / spawn CSRT tracker
#         det_box = [x1, y1, x2, y2]
#         best_tid, best_iou_val = None, 0.0
#         for tid, td in active_trackers.items():
#             v = iou(det_box, td["box"])
#             if v > best_iou_val:
#                 best_iou_val, best_tid = v, tid

#         if best_iou_val >= TRACKER_IOU_THRESH and best_tid is not None:
#             t = make_tracker()
#             t.init(frame, (x1, y1, x2-x1, y2-y1))
#             active_trackers[best_tid].update({"tracker": t, "box": det_box, "fails": 0, "score": score_value})
#             log_frame(f"  TRACKER {best_tid}: RE-INIT (IoU={best_iou_val:.2f})")
#         else:
#             t = make_tracker()
#             t.init(frame, (x1, y1, x2-x1, y2-y1))
#             active_trackers[NEXT_CSRT_ID] = {"tracker": t, "box": det_box, "fails": 0, "score": score_value}
#             log_frame(f"  TRACKER {NEXT_CSRT_ID}: SPAWNED")
#             NEXT_CSRT_ID += 1

#         # Draw + save
#         cv2.rectangle(annotated_frame, (x1c, y1c), (x2c, y2c), (0, 255, 0), 6)
#         cv2.putText(annotated_frame, f"helmet {score_value:.2f}",
#                     (x1c, max(30, y1c-10)),
#                     cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 4)

#         path = save_crop(frame, x1c, y1c, x2c, y2c,
#                          HELMET_CROP_DIR, "helmet",
#                          frame_count, score_value, ts)
#         if path:
#             log_frame(f"  HELMET CROP SAVED: {os.path.basename(path)} | conf={score_value:.4f}")
#             crop_count += 1
#         else:
#             log_frame("  HELMET CROP EMPTY.")

#         all_detections.append({
#             "frame": frame_count, "timestamp": ts,
#             "score": score_value, "label": "helmet",
#             "box": [x1c, y1c, x2c, y2c], "center": [cx, cy], "crop_path": path
#         })
#         total_helmet_det += 1
#         log_frame(f"  HELMET DETECTION ACCEPTED | conf={score_value:.4f}")

#     # =====================================================
#     # FRAME SUMMARY
#     # =====================================================

#     log_frame(
#         f"FRAME {frame_count} SUMMARY | "
#         f"RAW: {total_raw} | HELMET: {total_helmet_det} | "
#         f"NO_HELMET: {total_nohelmet_det} | "
#         f"TRACKERS: {len(active_trackers)} | TOTAL CROPS: {crop_count}"
#     )

#     with open(os.path.join(FRAME_DEBUG_DIR, f"frame_{frame_count:06d}_debug.txt"), "w", encoding="utf-8") as f:
#         f.write("\n".join(frame_lines) + "\n")

#     out.write(annotated_frame)
#     frame_count += 1

#     if frame_count % 10 == 0:
#         logger.info(f"PROCESSED {frame_count}/{total_frames} FRAMES")

# # =========================================================
# # RELEASE + SAVE
# # =========================================================

# cap.release()
# out.release()

# shutil.copy(OUTPUT_VIDEO_PATH, FINAL_SAVE_PATH)

# with open(DETECTIONS_JSON_PATH, "w", encoding="utf-8") as f:
#     json.dump(all_detections, f, indent=2)

# logger.info("INFERENCE COMPLETED SUCCESSFULLY")
# logger.info(f"OUTPUT VIDEO:    {FINAL_SAVE_PATH}")
# logger.info(f"TOTAL CROPS:     {crop_count}")
# logger.info(f"HELMET CROPS:    {HELMET_CROP_DIR}")
# logger.info(f"NO_HELMET CROPS: {NO_HELMET_CROP_DIR}")
# logger.info(f"DETECTIONS JSON: {DETECTIONS_JSON_PATH}")


[2026-05-18 11:05:16] USING DEVICE: cuda
INFO:groundingdino_laketown:USING DEVICE: cuda
[2026-05-18 11:05:16] LOADING PROCESSOR...
INFO:groundingdino_laketown:LOADING PROCESSOR...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[2026-05-18 11:05:19] PROCESSOR LOADED.
INFO:groundingdino_laketown:PROCESSOR LOADED.
[2026-05-18 11:05:19] LOADING MODEL...
INFO:groundingdino_laketown:LOADING MODEL...


model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

[2026-05-18 11:05:34] MODEL LOADED.
INFO:groundingdino_laketown:MODEL LOADED.
[2026-05-18 11:05:34] ROI POINTS: [[0, 1215], [297, 1004], [2064, 1071], [2067, 1510], [13, 1515]]
INFO:groundingdino_laketown:ROI POINTS: [[0, 1215], [297, 1004], [2064, 1071], [2067, 1510], [13, 1515]]


Exception: Could not open video: /content/drive/MyDrive/indian_helmet_detection_dataset/laketown_15sec.mp4